<a href="https://colab.research.google.com/github/thanh081205/Delivery-Routing-Optimization/blob/main/main_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### 1. Cài đặt thư viện

In [1]:
!pip install osmnx folium pgmpy joblib scikit-learn matplotlib pandas networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 9.8 MB/s eta 0:00:00


#### 2. Clone repo từ GitHub

In [2]:
import os

REPO_URL = "https://github.com/thanh081205/Delivery-Routing-Optimization.git"
REPO_NAME = "Delivery-Routing-Optimization"

# Clone repo nếu chưa có, pull nếu đã có
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print("✅ Clone repo thành công!")
else:
    !git -C {REPO_NAME} pull origin main
    print("✅ Pull repo thành công!")

# Thêm repo vào Python path để import được các module
import sys
sys.path.insert(0, f"/content/{REPO_NAME}")
print(f"✅ Đã thêm {REPO_NAME} vào Python path!")

Cloning into 'Delivery-Routing-Optimization'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 172 (delta 67), reused 102 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (172/172), 816.16 KiB | 5.20 MiB/s, done.
Resolving deltas: 100% (67/67), done.
✅ Clone repo thành công!
✅ Đã thêm Delivery-Routing-Optimization vào Python path!


#### 3. Load bản đồ khu vực trường Đại học Bách Khoa

In [3]:
from modules.graph.map_loader import load_map

LAT  = 10.7729
LON  = 106.6578
DIST = 1000  # bán kính 1km

graph_data = load_map(LAT, LON, DIST)

G     = graph_data["G"]
nodes = graph_data["nodes"]
edges = graph_data["edges"]

print(f"✅ Load bản đồ thành công!")
print(f"   Số nodes: {len(nodes)}")
print(f"   Số edges: {len(edges)}")

Đang tải bản đồ tại tọa độ (10.7729, 106.6578), bán kính 1000m...
✅ Tải xong! Số nút: 788, Số cạnh: 1864
✅ Load bản đồ thành công!
   Số nodes: 788
   Số edges: 1864


#### 4. Khởi tạo MapGraph và DeliveryVehicle

In [4]:
from modules.graph.core_system import MapGraph, DeliveryVehicle

# Khởi tạo MapGraph từ dữ liệu bản đồ
map_graph = MapGraph(graph_data)
print(f"✅ Khởi tạo MapGraph thành công!")
print(f"   Số node trong đồ thị: {len(map_graph.G.nodes)}")
print(f"   Số cạnh trong đồ thị: {len(map_graph.G.edges)}")

# Lấy node gần trường Bách Khoa nhất làm điểm xuất phát
origin_node = list(map_graph.G.nodes)[0]

# Lấy 3 node bất kỳ làm điểm giao hàng (sẽ thay bằng địa chỉ thực tế sau)
all_nodes = list(map_graph.G.nodes)
dest_A = all_nodes[10]
dest_B = all_nodes[20]
dest_C = all_nodes[30]
destinations = [dest_A, dest_B, dest_C]

# Khởi tạo xe giao hàng
# Xuất phát lúc 8h sáng (480 phút), tải trọng 2.5 tấn
vehicle = DeliveryVehicle(
    start_node=origin_node,
    start_time=480.0,
    capacity=2.5
)

# Thêm các điểm giao hàng với khung giờ
vehicle.add_delivery_point(dest_A, time_window=(480.0, 510.0))  # giao trong 30 phút đầu
vehicle.add_delivery_point(dest_B, time_window=(490.0, 540.0))  # giao từ phút 10 đến 60
vehicle.add_delivery_point(dest_C, time_window=(500.0, 560.0))  # giao từ phút 20 đến 80

print(f"\n✅ Khởi tạo DeliveryVehicle thành công!")
print(f"   Điểm xuất phát : {origin_node}")
print(f"   Điểm giao hàng : {destinations}")
print(f"   Giờ xuất phát  : {vehicle.current_time} phút (= 8:00 sáng)")
print(f"   Tải trọng xe   : {vehicle.capacity} tấn")

✅ Khởi tạo MapGraph thành công!
   Số node trong đồ thị: 788
   Số cạnh trong đồ thị: 1864

✅ Khởi tạo DeliveryVehicle thành công!
   Điểm xuất phát : 366370938
   Điểm giao hàng : [366380528, 366385478, 366397569]
   Giờ xuất phát  : 480.0 phút (= 8:00 sáng)
   Tải trọng xe   : 2.5 tấn


#### 5. Lọc đồ thị theo luật logic giao thông

In [5]:
from modules.bayes_logic.logic_filter import filter_graph

# Lọc các cạnh vi phạm luật giao thông
# vehicle.capacity = 2.5 tấn
cleaned_graph = filter_graph(graph_data, vehicle_weight=vehicle.capacity)

# Cập nhật đồ thị đã lọc vào MapGraph
map_graph.apply_logic_filter(cleaned_graph)

# In thống kê kết quả lọc
stats = cleaned_graph.graph["logic_filter_stats"]
print(f"✅ Lọc đồ thị thành công!")
print(f"   Tải trọng xe             : {stats['vehicle_weight_ton']} tấn")
print(f"   Cạnh bị xóa (tổng)       : {stats['removed_total']}")
print(f"   - Độ dài không hợp lệ    : {stats['removed_invalid_length']}")
print(f"   - Bị cấm truy cập        : {stats['removed_restricted_access']}")
print(f"   - Vượt giới hạn tải trọng: {stats['removed_weight_limit']}")
print(f"   - Đường đóng/xây dựng   : {stats['removed_closed_or_construction']}")
print(f"   Số cạnh còn lại          : {stats['remaining_edges']}")


✅ Lọc đồ thị thành công!
   Tải trọng xe             : 2.5 tấn
   Cạnh bị xóa (tổng)       : 0
   - Độ dài không hợp lệ    : 0
   - Bị cấm truy cập        : 0
   - Vượt giới hạn tải trọng: 0
   - Đường đóng/xây dựng   : 0
   Số cạnh còn lại          : 1864


#### 6. Tính xác suất kẹt xe bằng mạng Bayes

In [6]:
from modules.bayes_logic.bayes_model import compute_congestion

# Thiết lập điều kiện môi trường
WEATHER     = "rain"   # Thời tiết: "rain" hoặc "clear"
TIME_OF_DAY = "peak"   # Khung giờ: "peak" hoặc "normal"

# Tính xác suất kẹt xe cho từng cạnh
congestion_df = compute_congestion(
    edges=edges,
    weather=WEATHER,
    time_of_day=TIME_OF_DAY
)

# Thêm thông tin môi trường vào congestion_df để TV3 dùng
congestion_df["weather"]     = WEATHER
congestion_df["time_of_day"] = TIME_OF_DAY

print(f"✅ Tính xác suất kẹt xe thành công!")
print(f"   Điều kiện : thời tiết={WEATHER}, khung giờ={TIME_OF_DAY}")
print(f"   Số cạnh   : {len(congestion_df)}")
print(f"\n📊 Thống kê xác suất kẹt xe:")
print(f"   Trung bình : {congestion_df['p_congestion'].mean():.4f}")
print(f"   Thấp nhất  : {congestion_df['p_congestion'].min():.4f}")
print(f"   Cao nhất   : {congestion_df['p_congestion'].max():.4f}")
print(f"\n5 cạnh có xác suất kẹt xe cao nhất:")
print(congestion_df.nlargest(5, 'p_congestion')[['u', 'v', 'p_congestion']].to_string(index=False))

✅ Tính xác suất kẹt xe thành công!
   Điều kiện : thời tiết=rain, khung giờ=peak
   Số cạnh   : 1864

📊 Thống kê xác suất kẹt xe:
   Trung bình : 0.2618
   Thấp nhất  : 0.2423
   Cao nhất   : 0.2808

5 cạnh có xác suất kẹt xe cao nhất:
        u           v  p_congestion
366370938   366419772        0.2808
366370938  9843908827        0.2808
366371375  2434647841        0.2808
366372500 11587627043        0.2808
366378916  3887793637        0.2808


#### 7. Dự đoán thời gian di chuyển bằng ML

In [7]:
from modules.ml.travel_time_predictor import predict_travel_time

# Dự đoán thời gian di chuyển cho từng cạnh
# Kết hợp dữ liệu edges (TV1) + congestion_df (TV4) → weighted_edges (TV3)
weighted_edges = predict_travel_time(
    edges=edges,
    congestion_df=congestion_df
)

print(f"✅ Dự đoán thời gian di chuyển thành công!")
print(f"   Số cạnh dự đoán: {len(weighted_edges)}")
print(f"\n📊 Thống kê thời gian di chuyển (phút):")
print(f"   Trung bình : {weighted_edges['travel_time_min'].mean():.4f}")
print(f"   Thấp nhất  : {weighted_edges['travel_time_min'].min():.4f}")
print(f"   Cao nhất   : {weighted_edges['travel_time_min'].max():.4f}")
print(f"\n5 cạnh có thời gian di chuyển cao nhất:")
print(weighted_edges.nlargest(5, 'travel_time_min')[['u', 'v', 'travel_time_min']].to_string(index=False))

# Lấy cột key từ edges gốc và merge vào weighted_edges
key_df = edges[["u", "v", "key"]].drop_duplicates(subset=["u", "v"], keep="first")
weighted_edges = weighted_edges.merge(key_df, on=["u", "v"], how="left")
weighted_edges["key"] = weighted_edges["key"].fillna(0).astype(int)

print("✅ Đã bổ sung cột key vào weighted_edges!")
print(weighted_edges.head())

linear_regression: MAE=0.0817, RMSE=0.1074, R2=0.6891
decision_tree: MAE=0.0660, RMSE=0.0927, R2=0.7687
Dataset source: map_loader (1864 edges)
Saved mock dataset to: /content/Delivery-Routing-Optimization/data/mock_edge_travel_time.csv
Saved model artifact to: /content/Delivery-Routing-Optimization/features/travel_time_model.pkl
Saved report artifacts to: /content/Delivery-Routing-Optimization/modules/ml/artifacts
Selected model: decision_tree with RMSE=0.0927
✅ Dự đoán thời gian di chuyển thành công!
   Số cạnh dự đoán: 1864

📊 Thống kê thời gian di chuyển (phút):
   Trung bình : 0.2698
   Thấp nhất  : 0.2012
   Cao nhất   : 1.3153

5 cạnh có thời gian di chuyển cao nhất:
         u           v  travel_time_min
2302073740  6791442358           1.3153
2498077593  6372499819           1.3153
5765019200  6751290372           1.3153
6751290372  5765019200           1.3153
 366372500 11587627043           0.7306
✅ Đã bổ sung cột key vào weighted_edges!
           u           v  travel_tim

#### 8. Cập nhật trọng số ML vào MapGraph

In [8]:
# Cập nhật thời gian di chuyển từ TV3 vào đồ thị
map_graph.update_edge_weights(weighted_edges)

print(f"✅ Cập nhật trọng số vào MapGraph thành công!")
print(f"   Số cạnh đã cập nhật: {len(weighted_edges)}")

# Kiểm tra thử một cạnh bất kỳ
u_sample = int(weighted_edges.iloc[0]['u'])
v_sample = int(weighted_edges.iloc[0]['v'])

edge_data = map_graph.G[u_sample][v_sample][0]
print(f"\n🔍 Kiểm tra cạnh ({u_sample} → {v_sample}):")
print(f"   travel_time_min: {edge_data.get('travel_time_min', 'N/A')}")
print(f"   length         : {edge_data.get('length', 'N/A')}")

✅ Cập nhật trọng số vào MapGraph thành công!
   Số cạnh đã cập nhật: 1864

🔍 Kiểm tra cạnh (366370938 → 5060404769):
   travel_time_min: 0.2303
   length         : 36.35935589896651


#### 9. Tìm lộ trình tối ưu bằng A* và CSP

In [9]:
from modules.search.astar import run_astar

# Chạy thuật toán A*
result = run_astar(
    cleaned_graph=map_graph.G,
    weighted_edges=weighted_edges,
    origin=origin_node,
    destinations=destinations,
    time_windows=vehicle.deliveries,
    start_time=vehicle.current_time
)

# Cập nhật trạng thái xe sau khi tìm được lộ trình
vehicle.update_state(
    new_location=result["visited_order"][-1] if result["visited_order"] else origin_node,
    time_spent=result["total_time_min"]
)

print(f"✅ Tìm lộ trình thành công!")
print(f"\n📍 Kết quả lộ trình:")
print(f"   Thứ tự giao hàng : {result['visited_order']}")
print(f"   Tổng thời gian   : {result['total_time_min']:.4f} phút")
print(f"   Số node trên lộ trình: {len(result['route'])}")
print(f"\n🚗 Trạng thái xe sau khi giao hàng:")
print(f"   Vị trí hiện tại : {vehicle.current_location}")
print(f"   Thời gian hiện tại: {vehicle.current_time:.2f} phút (= {vehicle.current_time/60:.2f} giờ)")

✅ Tìm lộ trình thành công!

📍 Kết quả lộ trình:
   Thứ tự giao hàng : [366380528, 366385478, 366397569]
   Tổng thời gian   : 20.0000 phút
   Số node trên lộ trình: 56

🚗 Trạng thái xe sau khi giao hàng:
   Vị trí hiện tại : 366397569
   Thời gian hiện tại: 500.00 phút (= 8.33 giờ)


#### 10. Visualization lộ trình tối ưu lên bản đồ

In [10]:
import folium

# Tạo bản đồ nền
m = folium.Map(location=[LAT, LON], zoom_start=15)

# Vẽ toàn bộ mạng lưới đường bộ (màu xám nhạt)
for _, row in edges.iterrows():
    u_data = nodes[nodes["osmid"] == row["u"]]
    v_data = nodes[nodes["osmid"] == row["v"]]
    if not u_data.empty and not v_data.empty:
        u_coord = (u_data.iloc[0]["y"], u_data.iloc[0]["x"])
        v_coord = (v_data.iloc[0]["y"], v_data.iloc[0]["x"])
        folium.PolyLine(
            [u_coord, v_coord],
            color="gray",
            weight=1,
            opacity=0.3
        ).add_to(m)

# Vẽ lộ trình tối ưu (màu đỏ đậm)
route = result["route"]
for i in range(len(route) - 1):
    u_data = nodes[nodes["osmid"] == route[i]]
    v_data = nodes[nodes["osmid"] == route[i + 1]]
    if not u_data.empty and not v_data.empty:
        u_coord = (u_data.iloc[0]["y"], u_data.iloc[0]["x"])
        v_coord = (v_data.iloc[0]["y"], v_data.iloc[0]["x"])
        folium.PolyLine(
            [u_coord, v_coord],
            color="red",
            weight=4,
            opacity=0.8
        ).add_to(m)

# Đánh dấu điểm xuất phát (màu xanh lá)
origin_data = nodes[nodes["osmid"] == origin_node].iloc[0]
folium.Marker(
    location=[origin_data["y"], origin_data["x"]],
    popup=f"Xuất phát: {origin_node}",
    icon=folium.Icon(color="green", icon="play", prefix="fa")
).add_to(m)

# Đánh dấu các điểm giao hàng (màu đỏ)
for i, dest in enumerate(result["visited_order"]):
    dest_data = nodes[nodes["osmid"] == dest].iloc[0]
    folium.Marker(
        location=[dest_data["y"], dest_data["x"]],
        popup=f"Điểm giao {i+1}: {dest}",
        icon=folium.Icon(color="red", icon=f"star", prefix="fa")
    ).add_to(m)

print(f"✅ Visualization thành công!")
print(f"   Đường xám : toàn bộ mạng lưới đường bộ")
print(f"   Đường đỏ  : lộ trình tối ưu")
print(f"   Ghim xanh : điểm xuất phát")
print(f"   Ghim đỏ   : các điểm giao hàng")
m

✅ Visualization thành công!
   Đường xám : toàn bộ mạng lưới đường bộ
   Đường đỏ  : lộ trình tối ưu
   Ghim xanh : điểm xuất phát
   Ghim đỏ   : các điểm giao hàng
